# TP3 — Deep Learning : classification automatique de produits e-commerce

Matière : **Les fondamentaux de l'IA**  
Dataset : **Fashion-MNIST (Zalando Research)**

Objectif business : réduire le taux d'erreur de catégorisation de **12 %** à **moins de 5 %**.

In [ ]:
# Installation (si nécessaire) + imports
import sys
import subprocess
from pathlib import Path

required = [
    "numpy",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "tensorflow",
    "pandas",
]

try:
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import tensorflow as tf
    from tensorflow import keras
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
    import pandas as pd
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *required])
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import tensorflow as tf
    from tensorflow import keras
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
    import pandas as pd

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

ASSETS_DIR = Path("../assets")
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("Environnement prêt.")

## Étape 1 — Charger et explorer les données produits Zalando

Chargement du dataset `fashion_mnist` et inspection de la distribution des catégories.

In [ ]:
# Chargement du dataset Fashion-MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = [
    "T-shirt/top", "Pantalon", "Pull", "Robe", "Manteau",
    "Sandale", "Chemise", "Sneaker", "Sac", "Bottine"
]

print("=== EXPLORATION DU CATALOGUE ===")
print(f"Train : {X_train.shape[0]} images de {X_train.shape[1]}x{X_train.shape[2]} pixels")
print(f"Test : {X_test.shape[0]} images")
print(f"Pixels : min={X_train.min()}, max={X_train.max()} (niveaux de gris 0-255)")
print(f"Categories : {len(class_names)}")

print("\nDistribution des categories (train) :")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f" {class_names[u]:12s} : {c:>5d} articles ({c/len(y_train)*100:.1f}%)")

# Visualisation : 1 exemple par catégorie
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = np.where(y_train == i)[0][0]
    ax.imshow(X_train[idx], cmap="gray")
    ax.set_title(class_names[i], fontsize=10)
    ax.axis("off")

plt.suptitle("Catalogue Zalando - 1 exemple par categorie", fontsize=13)
plt.tight_layout()
plt.savefig(ASSETS_DIR / "catalogue_samples.png", bbox_inches="tight")
plt.show()

### Réponses — Étape 1

1. **Le catalogue est-il équilibré ?** Oui, il est quasi parfaitement équilibré (environ 10 % par catégorie), ce qui limite le biais d'entraînement vers une classe majoritaire.
2. **Risque de confusion visuelle (Pull vs Chemise) ?** Oui. Des classes visuellement proches partagent des textures et silhouettes similaires, ce qui augmente le risque de confusion pour l'algorithme.
3. **Résolution 28x28 réaliste pour la production ?** Non, c'est une résolution de benchmark. En production Zalando on utiliserait des images bien plus riches (souvent 224x224 ou plus) pour préserver les détails fins.

## Étape 2 — Prétraitement des images

Normalisation des pixels et création des formats requis pour ML classique (`flatten`) et CNN (tenseurs 4D).

In [ ]:
# Normalisation [0, 255] -> [0.0, 1.0]
X_train_norm = X_train.astype("float32") / 255.0
X_test_norm = X_test.astype("float32") / 255.0
print(f"Apres normalisation : min={X_train_norm.min():.1f}, max={X_train_norm.max():.1f}")

# Format pour ML classique : vecteur de 784 pixels
X_train_flat = X_train_norm.reshape(-1, 784)
X_test_flat = X_test_norm.reshape(-1, 784)

# Format pour CNN : tenseur avec canal
X_train_cnn = X_train_norm.reshape(-1, 28, 28, 1)
X_test_cnn = X_test_norm.reshape(-1, 28, 28, 1)

print(f"Forme ML classique (flatten) : {X_train_flat.shape}")
print(f"Forme reseau dense          : {X_train_norm.shape}")
print(f"Forme CNN                  : {X_train_cnn.shape}")

### Réponse — Pourquoi normaliser ?

La normalisation stabilise les gradients et accélère la convergence : les poids apprennent sur une échelle homogène, ce qui rend l'optimisation plus efficace. En gardant des pixels bruts 0-255, les mises à jour deviennent moins stables, l'entraînement converge plus lentement et le risque de mauvais minimum local augmente.

## Étape 3 — Baseline : Random Forest (ML classique)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf.fit(X_train_flat, y_train)
y_pred_rf = rf.predict(X_test_flat)
acc_rf = accuracy_score(y_test, y_pred_rf)
err_rf = 1 - acc_rf

print(f"Random Forest - Accuracy : {acc_rf*100:.2f}%")
print(f"Random Forest - Taux d'erreur : {err_rf*100:.2f}%")
print("Objectif business (< 5% erreur) atteint ?", "Oui" if err_rf < 0.05 else "Non")

### Réponses — Étape 3

1. **Accuracy suffisante pour l'objectif business ?** On vérifie via le taux d'erreur `(1 - accuracy)` : l'objectif est **strictement < 5 %**.
2. **Random Forest comprend-il la structure spatiale ?** Non. Il traite les pixels comme des variables indépendantes et n'exploite pas explicitement les relations locales (contours, formes, textures).

## Étape 4 — Réseau de neurones dense (MLP)

Architecture imposée : `Input(28,28) -> Flatten -> Dense(128,relu) -> Dense(64,relu) -> Dense(10,softmax)`.

In [ ]:
model_dense = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])

model_dense.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model_dense.summary()

history_dense = model_dense.fit(
    X_train_norm,
    y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.15,
    verbose=1,
)

### Réponses — Étape 4

1. **Nombre de paramètres entraînables** : donné par `model_dense.summary()` (nettement supérieur à 784 pixels), ce qui augmente la capacité d'apprentissage.
2. **Rôle de softmax** : transforme les logits en probabilités sur 10 classes (somme = 1), adapté à une classification multi-classe exclusive.
3. **Pourquoi `sparse_categorical_crossentropy` ?** Les labels sont des entiers (`0..9`), contrairement à `binary_crossentropy` qui cible une tâche binaire.

## Étape 5 — Réseau convolutif (CNN)

Architecture imposée : 2 blocs `Conv2D + MaxPooling` (32 puis 64 filtres), puis `Flatten -> Dense(64) -> Dropout(0.3) -> Dense(10, softmax)`.

In [ ]:
print(f"Forme pour CNN : {X_train_cnn.shape}")

model_cnn = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(32, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation="softmax"),
])

model_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model_cnn.summary()

history_cnn = model_cnn.fit(
    X_train_cnn,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.15,
    verbose=1,
)

### Réponses — Étape 5

1. **Pourquoi 26x26 après `Conv2D(32, 3x3)` sur 28x28 ?** Sans padding, le filtre 3x3 ne peut pas sortir du bord : `28 - 3 + 1 = 26`.
2. **Intérêt de `MaxPooling2D`** : réduction spatiale, robustesse locale et coût de calcul plus faible ; sans pooling, plus de paramètres et plus de surapprentissage.
3. **Pourquoi `Dropout(0.3)` réduit l'overfitting ?** Il empêche la co-adaptation excessive de neurones en imposant une régularisation stochastique à l'entraînement.

## Étape 6 — Courbes d'apprentissage : diagnostic overfitting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MLP
axes[0].plot(history_dense.history["accuracy"], "b-", label="Train")
axes[0].plot(history_dense.history["val_accuracy"], "r-", label="Validation")
axes[0].set_title("Reseau Dense (MLP)")
axes[0].set_xlabel("Epoque")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# CNN
axes[1].plot(history_cnn.history["accuracy"], "b-", label="Train")
axes[1].plot(history_cnn.history["val_accuracy"], "r-", label="Validation")
axes[1].set_title("CNN")
axes[1].set_xlabel("Epoque")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Courbes d'apprentissage - Diagnostic overfitting", fontsize=13)
plt.tight_layout()
plt.savefig(ASSETS_DIR / "learning_curves.png", bbox_inches="tight")
plt.show()

### Diagnostic overfitting

- Le **MLP** présente en général un écart train/validation plus visible après plusieurs époques, signe d'un début de surapprentissage.
- Le **CNN** converge mieux et maintient souvent un meilleur compromis généralisation/performance grâce aux convolutions + pooling + dropout.
- En production, on retiendrait le nombre d'époques juste avant la dégradation de `val_accuracy` (early stopping recommandé).

## Étape 7 — Comparaison des 3 approches et décision

In [ ]:
# Evaluation test
loss_dense, acc_dense = model_dense.evaluate(X_test_norm, y_test, verbose=0)
loss_cnn, acc_cnn = model_cnn.evaluate(X_test_cnn, y_test, verbose=0)

comparison_df = pd.DataFrame(
    {
        "Modele": ["Random Forest", "Reseau Dense (MLP)", "CNN"],
        "Accuracy": [acc_rf, acc_dense, acc_cnn],
    }
)
comparison_df["Tangens erreur (1 - Acc)"] = 1 - comparison_df["Accuracy"]
comparison_df["Passe seuil 5%"] = comparison_df["Tangens erreur (1 - Acc)"] < 0.05

print("=== COMPARAISON DES 3 APPROCHES ===")
display(comparison_df)

best_model_name = comparison_df.sort_values("Accuracy", ascending=False).iloc[0]["Modele"]
print(f"\nMeilleur modele (accuracy) : {best_model_name}")
print("Objectif business : taux d'erreur < 5%")

## Étape 8 — Matrice de confusion (CNN) + TOP-5 confusions

In [ ]:
# Predictions CNN
y_pred_cnn_proba = model_cnn.predict(X_test_cnn, verbose=0)
y_pred_cnn = np.argmax(y_pred_cnn_proba, axis=1)

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_cnn)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.title("Matrice de confusion - CNN")
plt.ylabel("Categorie reelle")
plt.xlabel("Categorie predite")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "confusion_matrix_cnn.png", bbox_inches="tight")
plt.show()

print("=== RAPPORT PAR CATEGORIE (CNN) ===")
print(classification_report(y_test, y_pred_cnn, target_names=class_names))

# Top-5 paires confondues hors diagonale
cm_off_diag = cm.copy().astype(int)
np.fill_diagonal(cm_off_diag, 0)

pairs = []
for i in range(cm_off_diag.shape[0]):
    for j in range(cm_off_diag.shape[1]):
        if cm_off_diag[i, j] > 0:
            pairs.append((i, j, cm_off_diag[i, j]))

pairs = sorted(pairs, key=lambda x: x[2], reverse=True)[:5]

top5_confusions = pd.DataFrame(
    [
        {
            "Categorie reelle": class_names[i],
            "Predite comme": class_names[j],
            "Nombre d'erreurs": n,
        }
        for i, j, n in pairs
    ]
)

print("=== TOP-5 CONFUSIONS LES PLUS FREQUENTES ===")
display(top5_confusions)

## Étape 9 — Interprétabilité CNN : filtres et feature maps

In [ ]:
# 9a - Filtres de la premiere couche conv
first_conv_layer = model_cnn.layers[0]
filters, biases = first_conv_layer.get_weights()
print("Shape filtres :", filters.shape)  # (3, 3, 1, 32)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[:, :, 0, i], cmap="gray")
    ax.set_title(f"F{i+1}", fontsize=7)
    ax.axis("off")

plt.suptitle("Filtres appris - 1re couche Conv2D (3x3)", fontsize=13)
plt.tight_layout()
plt.savefig(ASSETS_DIR / "filtres_conv.png", bbox_inches="tight")
plt.show()

# 9b - Feature maps pour un Sneaker (classe 7)
# Keras 3: utiliser model_cnn.inputs est plus robuste que model_cnn.input
activation_model = keras.Model(inputs=model_cnn.inputs, outputs=model_cnn.layers[0].output)

sample_idx = np.where(y_test == 7)[0][0]
sample = X_test_cnn[sample_idx:sample_idx + 1]
activations = activation_model.predict(sample, verbose=0)
print("Shape activations :", activations.shape)  # (1, 26, 26, 32)

fig, axes = plt.subplots(1, 9, figsize=(16, 2.8))
axes[0].imshow(X_test[sample_idx], cmap="gray")
axes[0].set_title("Original", fontsize=9)
axes[0].axis("off")

for i in range(8):
    axes[i + 1].imshow(activations[0, :, :, i], cmap="viridis")
    axes[i + 1].set_title(f"Filtre {i+1}", fontsize=9)
    axes[i + 1].axis("off")

plt.suptitle(f"Activations CNN - {class_names[y_test[sample_idx]]}", fontsize=12)
plt.tight_layout()
plt.savefig(ASSETS_DIR / "activations_conv.png", bbox_inches="tight")
plt.show()

### Réponses — Étape 9

1. Les filtres appris montrent des détecteurs de contours, gradients et motifs locaux, ce qui est cohérent avec un premier niveau de perception visuelle.
2. Sur les feature maps, certains filtres activent la silhouette de la sneaker (zones de bord/semelle), d'autres répondent davantage au fond.
3. Ces visualisations sont utiles mais insuffisantes pour prouver l'absence de biais : en complément, il faut des tests de robustesse (fonds variés), analyses de performance par sous-groupes d'images et audits sur données de production.

## Étape 10 — Debrief comité technique (exactement 3 insights)

**Insight 1 — ML classique vs Deep Learning**  
**Observation :** Le CNN obtient une accuracy supérieure au Random Forest, avec un taux d'erreur plus faible face à l'objectif business (< 5 %).  
**Explication :** Le CNN exploite la structure spatiale des images (contours, textures, formes), alors que le Random Forest traite les pixels comme des variables isolées.  
**Recommandation :** Le surcoût de calcul du CNN est justifié pour la production, car le gain de qualité réduit les erreurs de catalogage et leur coût business.

**Insight 2 — Dense vs CNN**  
**Observation :** Le réseau dense améliore la baseline classique, mais reste en dessous du CNN.  
**Explication :** Les convolutions apprennent des motifs locaux réutilisables et invariants localement, ce que le MLP ne fait pas explicitement après flatten.  
**Recommandation :** Réserver le réseau dense aux cas simples/contraints (prototypage rapide, ressources limitées) ; privilégier CNN pour la classification image en production.

**Insight 3 — Fiabilité en production**  
**Observation :** Les erreurs se concentrent sur des paires visuellement proches (ex. Pull/Chemise, T-shirt/Chemise) et certaines prédictions erronées peuvent être confiantes.  
**Explication :** À résolution 28x28, une partie des détails discriminants (coupe fine, texture) est perdue, rendant certaines confusions structurellement difficiles.  
**Recommandation :** Déployer avec garde-fous : seuil de confiance, revue humaine sur cas incertains/confus, et montée en résolution des images d'entrée.